In [1]:
from functools import partial

import numpy as np
import torch
from frouros.detectors.data_drift import MMDStreaming
from frouros.utils.kernels import rbf_kernel

In [2]:
detector = MMDStreaming(window_size=10, kernel=partial(rbf_kernel, sigma=0.5))
detector

MMD(callbacks=[], data_type=NumericalData(output_type=<class 'numpy.float32'>), statistical_type=MultivariateData(im_check=<built-in function ge>))

In [3]:
with open("encoded/v3.1_autoencoder_clean.pt", "rb") as f:
    encodings_clean = torch.load(f).to("cpu")
print(f"Encodings shape: {encodings_clean.shape}")

Encodings shape: torch.Size([527, 200])


In [4]:
with open("encoded/v3.1_autoencoder_dirty.pt", "rb") as f:
    encodings_dirty = torch.load(f).to("cpu")
print(f"Encodings shape: {encodings_dirty.shape}")

Encodings shape: torch.Size([12, 200])


In [5]:
_ = detector.fit(encodings_clean.cpu().numpy())
# Warm up the detector
for sample in encodings_clean[-detector.window_size + 1 :]:
    distance, _ = detector.update(sample.cpu().numpy())
    if distance is not None:
        print(f"Distance: {distance}")

In [6]:
distances_clean = [detector.update(sample.cpu().numpy())[0] for sample in encodings_clean]
distances_clean = np.array(distances_clean)
max, min = distances_clean.max(), distances_clean.min()
print(f"Max clean: {max}, Min clean: {min}")
mean, std = distances_clean.mean(), distances_clean.std()
print(f"Mean clean: {mean}, Std clean: {std}")

Max clean: 0.8294349246519725, Min clean: 0.011670120360055259
Mean clean: 0.23180992386212873, Std clean: 0.15934761531650526


In [7]:
result, _ = detector.update(encodings_dirty[0].cpu().numpy())
if result.distance <= (mean + 5 * std):
    print("Data drift detected, distance:", result.distance)
else:
    print("No data drift detected, distance:", result.distance)

Data drift detected, distance: 0.6370800455161009
